In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import numpy as np
import pandas as pd


In [3]:
### Load the trained model,scaler and label encoder
model = load_model('model.h5')

## load the scaler and label encoder
with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

with open('one_hot_encoder_geography.pkl', 'rb') as file:
    one_hot_encoder = pickle.load(file)

In [4]:
### Example input data for prediction
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [5]:
### One hot encode the 'Geography' feature
geography_encoded = one_hot_encoder.transform([[input_data['Geography']]]).toarray()
geography_df = pd.DataFrame(geography_encoded, columns=one_hot_encoder.get_feature_names_out(['Geography']))


d:\Krish_Naik\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [6]:
geography_df.head()

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [14]:
### Convert input data to dataframe and combine with encoded geography features
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [16]:
print(input_df['Gender'])

0    Male
Name: Gender, dtype: object


In [ ]:
### It's already done the label encoding --> Because of this we are getting this error.
input_df['Gender'] = label_encoder.transform(input_df['Gender'])


ValueError: y contains previously unseen labels: 1

In [20]:
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [21]:
## Combine the encoded geography features with the rest of the input data
input_df = pd.concat([input_df.drop('Geography', axis=1), geography_df], axis=1)

In [22]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [23]:
### Scaling the input features
scaled_input = scaler.transform(input_df)
scaled_input

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [24]:
### Predict the probability of churn
prediction = model.predict(scaled_input)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step


array([[0.02897565]], dtype=float32)

In [25]:
### prediction is the probability of churn, we can set a threshold to classify it as churn or not churn
prediction_probability = prediction[0][0]

In [26]:
prediction_probability

np.float32(0.02897565)

In [27]:
if prediction_probability > 0.5:
    print("The customer is likely to churn.")
else:
    print("The customer is not likely to churn.")

The customer is not likely to churn.
